In [1]:
# CELL 1 - Imports & Setup

In [18]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import os
from sagemaker.inputs import TrainingInput
from sagemaker import image_uris, get_execution_role
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, confusion_matrix
)
 
session     = sagemaker.Session()
role        = get_execution_role()
bucket      = "prognostica-cancer-project"
prefix      = "lung-cancer"
S3_FEATURES = f"s3://{bucket}/features/"
s3_client   = boto3.client("s3")
 
print(f"Role   : {role}")
print(f"Bucket : {bucket}")
print(f"Source : {S3_FEATURES}")

Role   : arn:aws:iam::381491860224:role/LabRole
Bucket : prognostica-cancer-project
Source : s3://prognostica-cancer-project/features/


In [19]:
# CELL 2 - Verify S3 files exist

In [20]:
TRAIN_URI = S3_FEATURES + "lung_train.csv"
VAL_URI   = S3_FEATURES + "lung_val.csv"
TEST_URI  = S3_FEATURES + "lung_test.csv"
 
def verify_s3_uri(uri):
    bkt, key = uri.replace("s3://", "").split("/", 1)
    try:
        s3_client.head_object(Bucket=bkt, Key=key)
        print(f"Found: {uri}")
    except Exception:
        raise FileNotFoundError(f"\nNOT FOUND: {uri}\n  Re-run data_preparation_handoff.ipynb first.")
 
print("Verifying S3 paths...")
verify_s3_uri(TRAIN_URI)
verify_s3_uri(VAL_URI)
verify_s3_uri(TEST_URI)

Verifying S3 paths...
Found: s3://prognostica-cancer-project/features/lung_train.csv
Found: s3://prognostica-cancer-project/features/lung_val.csv
Found: s3://prognostica-cancer-project/features/lung_test.csv


In [21]:
# CELL 3 - Download, fix ALL encoding issues, re-upload
#   Fixes applied consistently across ALL splits:
#     1. lung_cancer: YES/NO -> 1/0
#     2. gender: M/F -> 1/0
#     3. boolean columns (age groups): True/False -> 1/0
#     4. any remaining string columns: label encoded
#     5. No header row, target column first

In [22]:
os.makedirs("data", exist_ok=True)
 
def download_s3_uri(uri, local_path):
    bkt, key = uri.replace("s3://", "").split("/", 1)
    s3_client.download_file(bkt, key, local_path)
    print(f"Downloaded -> {local_path}")
 
def clean_and_encode(df, target_col="lung_cancer"):
    # 1. Encode target
    if df[target_col].dtype == object:
        df[target_col] = df[target_col].map({"YES": 1, "NO": 0})
 
    # 2. Encode gender
    if "gender" in df.columns and df["gender"].dtype == object:
        df["gender"] = df["gender"].map({"M": 1, "F": 0})
 
    # 3. Convert boolean columns to int
    bool_cols = df.select_dtypes(include="bool").columns.tolist()
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype(int)
        print(f"  Boolean cols converted: {bool_cols}")
 
    # 4. Encode any remaining string columns
    for col in df.columns:
        if df[col].dtype == object:
            unique_vals = sorted(df[col].unique())
            mapping = {val: idx for idx, val in enumerate(unique_vals)}
            df[col] = df[col].map(mapping)
            print(f"  String col encoded: '{col}' -> {mapping}")
 
    # 5. Move target to first column
    cols = [target_col] + [c for c in df.columns if c != target_col]
    df   = df[cols]
 
    return df
 
def prepare_and_upload(local_path, s3_key, target_col="lung_cancer"):
    df = pd.read_csv(local_path)
    print(f"\nCleaning {local_path}...")
    df = clean_and_encode(df, target_col)
    out = local_path.replace(".csv", "_ready.csv")
    df.to_csv(out, index=False, header=False)
    s3_client.upload_file(out, bucket, s3_key)
    print(f"Uploaded -> s3://{bucket}/{s3_key}  ({df.shape[0]} rows x {df.shape[1]} cols)")
    return df
 
print("\nDownloading from S3...")
download_s3_uri(TRAIN_URI, "data/lung_train.csv")
download_s3_uri(VAL_URI,   "data/lung_val.csv")
download_s3_uri(TEST_URI,  "data/lung_test.csv")
 
print("\nEncoding & re-uploading...")
train_df = prepare_and_upload("data/lung_train.csv", f"{prefix}/train/lung_train.csv")
val_df   = prepare_and_upload("data/lung_val.csv",   f"{prefix}/val/lung_val.csv")
test_df  = prepare_and_upload("data/lung_test.csv",  f"{prefix}/test/lung_test.csv")
 
# Verify all columns are numeric
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    bad = [c for c in df.columns if df[c].dtype == object]
    print(f"{name} non-numeric cols: {bad if bad else 'None - all clean!'}")
 
train_input = TrainingInput(f"s3://{bucket}/{prefix}/train/", content_type="text/csv")
val_input   = TrainingInput(f"s3://{bucket}/{prefix}/val/",   content_type="text/csv")
 
print("\nData ready for training.")
 


Downloaded -> data/lung_train.csv
Downloaded -> data/lung_val.csv
Downloaded -> data/lung_test.csv

Encoding & re-uploading...

Cleaning data/lung_train.csv...
  Boolean cols converted: ['age_group_41-50', 'age_group_51-60', 'age_group_61-70', 'age_group_71+']
Uploaded -> s3://prognostica-cancer-project/lung-cancer/train/lung_train.csv  (13699 rows x 20 cols)

Cleaning data/lung_val.csv...
  Boolean cols converted: ['age_group_41-50', 'age_group_51-60', 'age_group_61-70', 'age_group_71+']
Uploaded -> s3://prognostica-cancer-project/lung-cancer/val/lung_val.csv  (2936 rows x 20 cols)

Cleaning data/lung_test.csv...
  Boolean cols converted: ['age_group_41-50', 'age_group_51-60', 'age_group_61-70', 'age_group_71+']
Uploaded -> s3://prognostica-cancer-project/lung-cancer/test/lung_test.csv  (2936 rows x 20 cols)
train non-numeric cols: None - all clean!
val non-numeric cols: None - all clean!
test non-numeric cols: None - all clean!

Data ready for training.


In [23]:
# Check what the raw files actually look like from S3
import pandas as pd

train_raw = pd.read_csv("data/lung_train.csv")
print("Shape:", train_raw.shape)
print("\nDtypes:")
print(train_raw.dtypes)
print("\nFirst 2 rows:")
print(train_raw.head(2))
print("\nTarget value counts:")
print(train_raw["lung_cancer"].value_counts())
print("\nAny NaN values:", train_raw.isna().sum().sum())

Shape: (13699, 20)

Dtypes:
gender                   object
age                       int64
smoking                   int64
yellow_fingers            int64
anxiety                   int64
peer_pressure             int64
chronic_disease           int64
fatigue                   int64
allergy                   int64
wheezing                  int64
alcohol_consuming         int64
coughing                  int64
shortness_of_breath       int64
swallowing_difficulty     int64
chest_pain                int64
lung_cancer              object
age_group_41-50            bool
age_group_51-60            bool
age_group_61-70            bool
age_group_71+              bool
dtype: object

First 2 rows:
  gender  age  smoking  yellow_fingers  anxiety  peer_pressure  \
0      M   60        1               2        1              2   
1      F   55        2               1        2              2   

   chronic_disease  fatigue  allergy  wheezing  alcohol_consuming  coughing  \
0                1       

In [62]:
from sklearn.model_selection import train_test_split

# ── Load full dataset ──────────────────────────────────────────
df = pd.read_csv("data/lung_model_data.csv")

# ── Fix all encoding ───────────────────────────────────────────
# 1. Target: YES/NO -> 1/0
df["lung_cancer"] = df["lung_cancer"].map({"YES": 1, "NO": 0})

# 2. Gender: M/F -> 1/0
df["gender"] = df["gender"].map({"M": 1, "F": 0})

# 3. Recode 1/2 features to 0/1 (all int cols except age)
recode_cols = ["smoking", "yellow_fingers", "anxiety", "peer_pressure",
               "chronic_disease", "fatigue", "allergy", "wheezing",
               "alcohol_consuming", "coughing", "shortness_of_breath",
               "swallowing_difficulty", "chest_pain"]
for col in recode_cols:
    df[col] = df[col] - 1

# 4. Drop redundant age group bool cols — age is already numeric
df = df.drop(columns=["age_group_41-50", "age_group_51-60",
                       "age_group_61-70", "age_group_71+"])

# ── Move target to first column ────────────────────────────────
cols = ["lung_cancer"] + [c for c in df.columns if c != "lung_cancer"]
df   = df[cols]

print("Shape:", df.shape)
print("\nValue counts:\n", df["lung_cancer"].value_counts())
print("\nFirst 3 rows:\n", df.head(3))
print("\nAll numeric:", df.select_dtypes(include="object").columns.tolist())

# ── Stratified split ───────────────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["lung_cancer"])
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["lung_cancer"])

print(f"\nTrain: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

# ── Upload to S3 ───────────────────────────────────────────────
for split_df, name, key in [
    (train_df, "train", f"{prefix}/train/lung_train.csv"),
    (val_df,   "val",   f"{prefix}/val/lung_val.csv"),
    (test_df,  "test",  f"{prefix}/test/lung_test.csv"),
]:
    out = f"data/lung_{name}_ready.csv"
    split_df.to_csv(out, index=False, header=False)
    s3_client.upload_file(out, bucket, key)
    print(f"Uploaded {name}: {split_df.shape}")

train_input = TrainingInput(f"s3://{bucket}/{prefix}/train/", content_type="text/csv")
val_input   = TrainingInput(f"s3://{bucket}/{prefix}/val/",   content_type="text/csv")
print("\nData ready!")

Shape: (19571, 16)

Value counts:
 lung_cancer
1    17012
0     2559
Name: count, dtype: int64

First 3 rows:
    lung_cancer  gender  age  smoking  yellow_fingers  anxiety  peer_pressure  \
0            1       1   69        1               0        0              1   
1            1       1   71        1               1        0              0   
2            0       1   61        1               0        0              1   

   chronic_disease  fatigue  allergy  wheezing  alcohol_consuming  coughing  \
0                0        1        0         0                  1         1   
1                1        0        1         1                  0         0   
2                1        0        1         1                  0         0   

   shortness_of_breath  swallowing_difficulty  chest_pain  
0                    1                      0           0  
1                    1                      1           0  
2                    1                      1           1  

All numeri

In [63]:

# CELL 4 - Fixed estimator with logloss + threshold tuning


In [64]:

yes_count        = (train_df.iloc[:, 0] == 1).sum()
no_count         = (train_df.iloc[:, 0] == 0).sum()
scale_pos_weight = round(yes_count / no_count, 4)
print(f"scale_pos_weight = {scale_pos_weight}  (YES: {yes_count} / NO: {no_count})")

container = image_uris.retrieve(
    framework="xgboost",
    region=session.boto_region_name,
    version="1.7-1",
)

xgb = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session,
    hyperparameters={
        "objective":         "binary:logistic",
        "eval_metric":       "logloss",
        "scale_pos_weight":  scale_pos_weight,
        "num_round":         100,
        "max_depth":         4,
        "min_child_weight":  1,
        "eta":               0.1,
        "subsample":         0.8,
        "colsample_bytree":  0.8,
    },
)
print("Estimator ready.")

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


scale_pos_weight = 6.6488  (YES: 11908 / NO: 1791)
Estimator ready.


In [65]:
# CELL 5 - Train with no validation channel

In [72]:

xgb.fit(
    inputs={"train": train_input},  # remove validation channel
    logs=True,
    wait=True,
)
print("\nTraining complete!")
print(f"Model artifact -> {xgb.model_data}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-22-20-47-41-882


2026-03-22 20:47:43 Starting - Starting the training job...
2026-03-22 20:47:57 Starting - Preparing the instances for training...
2026-03-22 20:48:45 Downloading - Downloading the training image......
2026-03-22 20:49:36 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-22 20:49:43.420 ip-10-0-73-166.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-22 20:49:43.483 ip-10-0-73-166.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-22:20:49:43:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-22:20:49:43:INFO] Failed to parse hyperparameter 

In [73]:
# CELL 6 - Batch Transform


In [74]:
# Upload test features only (no target col)
test_features = test_df.iloc[:, 1:]
test_features.to_csv("data/lung_test_features.csv", index=False, header=False)
s3_client.upload_file(
    "data/lung_test_features.csv", bucket,
    f"{prefix}/test_features/lung_test_features.csv"
)
print(f"Test features uploaded: {test_features.shape}")

transformer = xgb.transformer(
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/predictions",
    assemble_with="Line",
    accept="text/csv",
)

print(f"Using model: {xgb.model_data}")

transformer.transform(
    data=f"s3://{bucket}/{prefix}/test_features/",
    content_type="text/csv",
    split_type="Line",
    wait=True,
    logs=True,
)

print("\nBatch transform complete!")

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-03-22-20-50-29-631


Test features uploaded: (2936, 15)


INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-03-22-20-50-30-386


Using model: s3://prognostica-cancer-project/lung-cancer/output/sagemaker-xgboost-2026-03-22-20-47-41-882/output/model.tar.gz
.............................../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-22:20:55:39:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-22:20:55:39:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-22:20:55:39:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.

In [75]:
# CELL 7 - Evaluate


In [77]:
s3_client.download_file(
    bucket,
    f"{prefix}/predictions/lung_test_features.csv.out",
    "data/lung_predictions.csv"
)

probs  = pd.read_csv("data/lung_predictions.csv", header=None)[0]
labels = test_df.iloc[:, 0].reset_index(drop=True)

# Find optimal threshold
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(labels, probs)
optimal_idx       = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold: {optimal_threshold:.4f}  (default is 0.5)")

preds = (probs >= optimal_threshold).astype(int)

print(f"\nEvaluation Results (threshold={optimal_threshold:.4f})")
print(f"Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(labels, probs):.4f}")

print("\nConfusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(labels, preds),
    index=["Actual NO", "Actual YES"],
    columns=["Pred NO", "Pred YES"]
))

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["NO (0)", "YES (1)"]))

Optimal threshold: 0.9849  (default is 0.5)

Evaluation Results (threshold=0.9849)
Accuracy : 0.1693
ROC-AUC  : 0.4879

Confusion Matrix:
            Pred NO  Pred YES
Actual NO       368        16
Actual YES     2423       129

Classification Report:
              precision    recall  f1-score   support

      NO (0)       0.13      0.96      0.23       384
     YES (1)       0.89      0.05      0.10      2552

    accuracy                           0.17      2936
   macro avg       0.51      0.50      0.16      2936
weighted avg       0.79      0.17      0.11      2936

